##import 

In [10]:
import os
from dotenv import load_dotenv


In [11]:
load_dotenv()

True

In [12]:
groq_key = os.getenv("GROQ_API_KEY")

In [14]:
jina_key = os.getenv("JINA_API_KEY")

In [15]:
print("env variable loaded")

env variable loaded


##data loading

In [16]:
data_file_path = os.path.join("data","HR_Policies.txt")

In [17]:
data_file_path

'data\\HR_Policies.txt'

##data ingestion

In [18]:
from langchain_community.document_loaders import TextLoader

In [22]:
hr_policy_loader = TextLoader(data_file_path,encoding="utf-8",)

In [23]:
documents = hr_policy_loader.load()

In [27]:
print("Data Loaded")
print("="*40)
print(documents)

Data Loaded
[Document(metadata={'source': 'data\\HR_Policies.txt'}, page_content="\ufeffHUMAN RESOURCES POLICIES\n\nPurpose and Scope\nThese policies describe general workplace expectations for employees and managers. They are intended as a practical guide and should be read together with employment agreements, benefit documents, and applicable law. They should be adapted to the organization, its location, and its workforce. Where a policy conflicts with applicable law, the law takes precedence. HR is responsible for maintaining the current version and communicating material updates.\n\n1. Equal Opportunity, Inclusion, and Respect\nThe organization provides equal employment opportunity. Hiring, pay, promotion, training, discipline, and other employment decisions should be based on role requirements, qualifications, performance, and business needs. Discrimination, harassment, bullying, and retaliation are prohibited.\n\nEmployees should use respectful language, respect reasonable person

In [28]:
len(documents)

1

In [32]:
print(documents[0].page_content)

HUMAN RESOURCES POLICIES

Purpose and Scope
These policies describe general workplace expectations for employees and managers. They are intended as a practical guide and should be read together with employment agreements, benefit documents, and applicable law. They should be adapted to the organization, its location, and its workforce. Where a policy conflicts with applicable law, the law takes precedence. HR is responsible for maintaining the current version and communicating material updates.

1. Equal Opportunity, Inclusion, and Respect
The organization provides equal employment opportunity. Hiring, pay, promotion, training, discipline, and other employment decisions should be based on role requirements, qualifications, performance, and business needs. Discrimination, harassment, bullying, and retaliation are prohibited.

Employees should use respectful language, respect reasonable personal and cultural boundaries, and avoid conduct that creates an intimidating, hostile, or humiliat

In [33]:
print(documents[0].metadata)

{'source': 'data\\HR_Policies.txt'}


In [35]:
print(f"total number of char: len(documents[0].page_content)")

total number of char: len(documents[0].page_content)


In [36]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [62]:
document_splitter = RecursiveCharacterTextSplitter(
    chunk_size =1000,
    chunk_overlap = 100
)
chunks = document_splitter.split_documents(documents)

In [63]:
chunks

[Document(metadata={'source': 'data\\HR_Policies.txt'}, page_content='\ufeffHUMAN RESOURCES POLICIES\n\nPurpose and Scope\nThese policies describe general workplace expectations for employees and managers. They are intended as a practical guide and should be read together with employment agreements, benefit documents, and applicable law. They should be adapted to the organization, its location, and its workforce. Where a policy conflicts with applicable law, the law takes precedence. HR is responsible for maintaining the current version and communicating material updates.\n\n1. Equal Opportunity, Inclusion, and Respect\nThe organization provides equal employment opportunity. Hiring, pay, promotion, training, discipline, and other employment decisions should be based on role requirements, qualifications, performance, and business needs. Discrimination, harassment, bullying, and retaliation are prohibited.'),
 Document(metadata={'source': 'data\\HR_Policies.txt'}, page_content='Employees

In [64]:
print(chunks[0])

page_content='﻿HUMAN RESOURCES POLICIES

Purpose and Scope
These policies describe general workplace expectations for employees and managers. They are intended as a practical guide and should be read together with employment agreements, benefit documents, and applicable law. They should be adapted to the organization, its location, and its workforce. Where a policy conflicts with applicable law, the law takes precedence. HR is responsible for maintaining the current version and communicating material updates.

1. Equal Opportunity, Inclusion, and Respect
The organization provides equal employment opportunity. Hiring, pay, promotion, training, discipline, and other employment decisions should be based on role requirements, qualifications, performance, and business needs. Discrimination, harassment, bullying, and retaliation are prohibited.' metadata={'source': 'data\\HR_Policies.txt'}


In [65]:
print(chunks[1])

page_content='Employees should use respectful language, respect reasonable personal and cultural boundaries, and avoid conduct that creates an intimidating, hostile, or humiliating environment. Harassment can include unwelcome verbal, physical, written, or online conduct. Employees who experience or witness inappropriate conduct should report it to a manager, HR, or another designated contact. Reports will be reviewed fairly and promptly, with information shared only as needed to address the concern. Retaliation for a good-faith report or participation in a review is prohibited.

The organization will consider reasonable workplace accommodations as required by law. Employees seeking an accommodation should contact HR or their manager and provide information needed to discuss an effective arrangement.' metadata={'source': 'data\\HR_Policies.txt'}


In [66]:
print(chunks[5])

page_content='6. Leave and Time Off

6.1 Sample Policy Notice
The leave balances and rules in this section are fictional sample values for demonstration. They are not a statement of legal entitlement. Before adopting this policy, HR must check local law, employment agreements, and applicable benefit plans. Where a law or agreement provides a greater entitlement, that entitlement applies. Unless stated otherwise, one leave day means one scheduled workday for the employee.' metadata={'source': 'data\\HR_Policies.txt'}


##embedings

In [67]:
from langchain_community.embeddings import JinaEmbeddings

In [68]:
embeding_model =JinaEmbeddings( model_name="jina-embeddings-v2-base-en")

In [69]:
embeding_model.model_name

'jina-embeddings-v2-base-en'

#store data in vector databasee

In [70]:
from langchain_community.vectorstores import FAISS

In [71]:
vector_store = FAISS.from_documents(chunks,embeding_model)
print(f"chunks are stored: {vector_store.index.ntotal}")

chunks are stored: 26


In [77]:
test_query ="Do unused sick or casual leave days carry forward?"

In [78]:
top_matches = vector_store.similarity_search(test_query,k=2)

In [79]:
print(top_matches)

[Document(id='2b2f1fa5-1dbd-4f33-b25c-41432296c1f0', metadata={'source': 'data\\HR_Policies.txt'}, page_content='The sample allowance is 10 working days per leave year. Unused sick leave does not carry forward and is not paid out, except where law or an applicable plan requires otherwise. If the allowance is exhausted, employees may request available annual leave, another legally protected leave, or unpaid leave. The organization will consider accommodation requests and protected leave rights as required by law.'), Document(id='7ba5f487-21b5-48dd-b5fe-33092a5a5cf8', metadata={'source': 'data\\HR_Policies.txt'}, page_content='6.12 Examples of Balance and Carry-Forward\nExample A: An employee takes 12 annual leave days during the year and has accrued all 18 days. The remaining 6 annual leave days may be carried into the next year, subject to the 31 March use-by date.\n\nExample B: An employee has 12 unused annual leave days on 31 December. Under this sample rule, 8 days carry forward and

In [80]:
for i,match in enumerate(top_matches,start=1):
    print(f"----MATCH{i}-----")
    print(match.page_content)
    print()

----MATCH1-----
The sample allowance is 10 working days per leave year. Unused sick leave does not carry forward and is not paid out, except where law or an applicable plan requires otherwise. If the allowance is exhausted, employees may request available annual leave, another legally protected leave, or unpaid leave. The organization will consider accommodation requests and protected leave rights as required by law.

----MATCH2-----
6.12 Examples of Balance and Carry-Forward
Example A: An employee takes 12 annual leave days during the year and has accrued all 18 days. The remaining 6 annual leave days may be carried into the next year, subject to the 31 March use-by date.

Example B: An employee has 12 unused annual leave days on 31 December. Under this sample rule, 8 days carry forward and must be used by 31 March; the other 4 days expire unless law or an approved exception provides otherwise.

Example C: An employee has 4 unused sick days and 2 unused casual days on 31 December. The

## TOOL

In [112]:
retriever = vector_store.as_retriever(search_kwargs={"k":3})
def search_hr_policy(question:str)->str:
    """
    search the HR policy documents for information about leaves, probation and so on
    """
    
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

### DATA Retrieval 

In [81]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model ="openai/gpt-oss-120b",
    temperature=0
)

In [82]:
llm.model_name

'openai/gpt-oss-120b'

In [88]:
test_query ="what is the rajdhani of india?reply in funny way"
test_response = llm.invoke(test_query)

In [90]:
answer = test_response.content.replace("\u202f", " ")
print(answer)

Sure thing! 🎉

The “Rajdhani” (capital) of India is **New Delhi**—the city that’s basically the *VIP lounge* for the whole country. Think of it as India’s ultimate “headquarters,” where the Prime Minister’s office has the best coffee, the Parliament has the most dramatic debates, and the traffic… well, let’s just say it’s a *real‑life* video‑game level! 🚦🏎️

So if you ever need a place to practice your “I‑am‑the‑boss” walk, just head to New Delhi—just watch out for the auto‑rickshaws, they’re faster than a Bollywood dance number! 😄🕺💃


## AI AGENT

In [92]:
from langchain.agents import create_agent

In [113]:
hr_agent = create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt="""
    You are a friendly HR assistant.
    Always use search_hr_policy tool to look up
    facts before answering.
    If the answer isnot in the search result, say you don't know
    instead of guessing""")


print("HR assistant is ready to answer....!")

HR assistant is ready to answer....!


In [95]:
def ask_hr_assistant(question: str)-> str:
    """Send question to the RAG agent and print a nicely formatted answer"""
    print("="*60)
    print("Question:",question)
    print("-"*60)
    
    response = hr_agent.invoke({"messages":[{"role":"user","content":question}]})
    answer = response["messages"][-1].content
    print("ANSWER:",answer)
    print("="*60)
    print()

In [114]:
response = hr_agent.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"tell me about the leave policy?"
            }
        ]
    }
)


In [118]:
response["messages"][-1].content

'**Company Leave Policy (summary of the current HR policy)**  \n\n| Aspect | What the policy says |\n|--------|----------------------|\n| **General framework** | The policy provides sample (illustrative) leave balances and rules. It is not a legal entitlement statement; any higher entitlement required by law, an employment contract, or a benefit plan overrides the sample values. |\n| **Accrual & eligibility** | • Leave begins to accrue from an employee’s start date. <br>• Employees may use leave after **30\u202fcalendar days** of employment, provided they have enough accrued balance and receive manager approval. <br>• No employee may have a negative leave balance unless HR grants a written exception. |\n| **Part‑time employees** | Leave is **pro‑rated** based on the employee’s scheduled working days or hours. |\n| **Employees joining or leaving mid‑year** | Their annual leave allowance is **pro‑rated** for the portion of the leave year they actually work. HR calculates the balance and 

In [119]:
response = hr_agent.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"tell me RAjdhani of india?"
            }
        ]
    }
)


In [120]:
response["messages"][-1].content

'I’m sorry, but I don’t have that information.'

### NA